In [ ]:

words=open('names.txt','r').read().splitlines()
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [18]:
stoi={}
stoi['.']=0
# ord: char->int 
# chr: int->char
for i in range(26):
    stoi[chr(i+ord('a'))]=i+1
stoi.items()

dict_items([('.', 0), ('a', 1), ('b', 2), ('c', 3), ('d', 4), ('e', 5), ('f', 6), ('g', 7), ('h', 8), ('i', 9), ('j', 10), ('k', 11), ('l', 12), ('m', 13), ('n', 14), ('o', 15), ('p', 16), ('q', 17), ('r', 18), ('s', 19), ('t', 20), ('u', 21), ('v', 22), ('w', 23), ('x', 24), ('y', 25), ('z', 26)])

In [19]:
# 易错点: stoi.items() 不能是stoi,stoi就只遍历key了
itos={i:s for s,i in stoi.items()}
itos.items()

dict_items([(0, '.'), (1, 'a'), (2, 'b'), (3, 'c'), (4, 'd'), (5, 'e'), (6, 'f'), (7, 'g'), (8, 'h'), (9, 'i'), (10, 'j'), (11, 'k'), (12, 'l'), (13, 'm'), (14, 'n'), (15, 'o'), (16, 'p'), (17, 'q'), (18, 'r'), (19, 's'), (20, 't'), (21, 'u'), (22, 'v'), (23, 'w'), (24, 'x'), (25, 'y'), (26, 'z')])

In [133]:
import torch

# 构造训练集


xs,ys=[],[]
for word in words:
    chs=['.']+list(word)+['.']
    for ch1,ch2 in zip(chs,chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs=torch.tensor(xs)
ys=torch.tensor(ys)


In [71]:
xs[:]

tensor([ 0,  5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9,
        19,  1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1])

In [134]:
# one-hot encoding 
import torch.nn.functional as F 
xenc=F.one_hot(xs,num_classes=27).float()
# random seed
g=torch.Generator().manual_seed(114514)
# init params
W=torch.randn((27,27),generator=g,requires_grad=True)
# forward pass
counts=xenc@W.exp()
# transform to probability model
probs=counts/counts.sum(dim=1,keepdim=True)
probs[:2,:]

tensor([[0.0273, 0.0151, 0.0226, 0.0268, 0.0215, 0.0287, 0.0078, 0.0566, 0.0166,
         0.0748, 0.0366, 0.0029, 0.0116, 0.0166, 0.0194, 0.0532, 0.0046, 0.0146,
         0.0264, 0.0919, 0.0656, 0.0848, 0.1574, 0.0372, 0.0563, 0.0071, 0.0161],
        [0.0110, 0.0097, 0.0136, 0.0243, 0.0530, 0.0596, 0.0135, 0.0346, 0.0201,
         0.0058, 0.0208, 0.0188, 0.0070, 0.0576, 0.0268, 0.0394, 0.0230, 0.0142,
         0.0043, 0.0016, 0.4191, 0.0086, 0.0109, 0.0210, 0.0060, 0.0151, 0.0606]],
       grad_fn=<SliceBackward0>)

In [90]:

probs.sum(dim=1,keepdim=True)[:5]

tensor([[1.0000],
        [1.0000],
        [1.0000],
        [1.0000],
        [1.0000]])

In [73]:
ys

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [97]:
print(probs[0,5],probs[1,13],probs[2,13],probs[3,1])

tensor(0.0287, grad_fn=<SelectBackward0>) tensor(0.0576, grad_fn=<SelectBackward0>) tensor(0.0664, grad_fn=<SelectBackward0>) tensor(0.0047, grad_fn=<SelectBackward0>)


In [141]:
for i in range(1001):
    # forward pass
    counts=xenc@W.exp()
    # transform to probability model
    probs=counts/counts.sum(dim=1,keepdim=True)
    loss=-probs[torch.arange(len(ys)),ys].log().mean()+0.01*(W**2).mean()
    W.grad=None
    loss.backward() 
    W.data-=50*W.grad
    if i%100==0:
        print(loss)

tensor(2.4891, grad_fn=<AddBackward0>)
tensor(2.4845, grad_fn=<AddBackward0>)
tensor(2.4824, grad_fn=<AddBackward0>)
tensor(2.4814, grad_fn=<AddBackward0>)
tensor(2.4810, grad_fn=<AddBackward0>)
tensor(2.4807, grad_fn=<AddBackward0>)
tensor(2.4806, grad_fn=<AddBackward0>)
tensor(2.4805, grad_fn=<AddBackward0>)
tensor(2.4805, grad_fn=<AddBackward0>)
tensor(2.4805, grad_fn=<AddBackward0>)
tensor(2.4804, grad_fn=<AddBackward0>)


In [162]:
for i in range(10):
    ix=0
    out=[]
    while True:
        p=W[ix].exp()
        p=p/p.sum()
        ix=torch.multinomial(p,num_samples=1,replacement=True,generator=g).item()
        out.append(itos[ix])
        if ix==0:
            break
    print(''.join(out))

lyna.
jarwja.
kime.
bbe.
kutzayly.
pagaliaryvikafry.
ppa.
gamxorelyl.
kilal.
diaslah.
